<a href="https://colab.research.google.com/github/beingadithsagar-jpg/Assignment_1and_2/blob/main/%20%20%20assignment-2/agent_loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [1]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

API key loaded from Colab Secrets.


In [2]:
# Quiet ensures your UI is not full of logs.
!pip install -q google-genai
print("Done.")

Done.


In [3]:
# @title User input needed.

# TODO: Fill in documents in the format mentioned in the first example.
DOCUMENTS = {
    "sports": """
    Students can share documents about cricket, football and basketball.
    They can discuss match results, player performances and tournament records.
    Documents may include team statistics and training techniques.
    Students can also share information about upcoming sporting events.
    """,

    "manga": """
    Students can share documents about popular manga series.
    They can discuss characters, storylines and different manga genres.
    Documents may include manga recommendations and author information.
    Students can also compare different manga series.
    """,

    "food": """
    Students can share documents about different cuisines.
    They can discuss recipes, cooking techniques and traditional dishes.
    Documents may include nutritional information and food cultures.
    Students can also share their favourite restaurants and food recommendations.
    """,

    "music": """
    Students can share documents about different music genres.
    They can discuss artists, albums and musical instruments.
    Documents may include music recommendations and concert information.
    Students can also share their favourite songs and playlists.
    """
}

In [4]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_documents("students share documents"))


[Source: sports]

    Students can share documents about cricket, football and basketball.
    They can discuss match results, player performances and tournament records.
    Documents may include team statistics and training techniques.
    Students can also share information about upcoming sporting events.
    

[Source: music]

    Students can share documents about different music genres.
    They can discuss artists, albums and musical instruments.
    Documents may include music recommendations and concert information.
    Students can also share their favourite songs and playlists.
    


In [5]:
# @title Ask gemini to search for something
# Configure the Gemini model

MODEL_NAME = "gemini-3.5-flash"

print("Model ready:", MODEL_NAME)

Model ready: gemini-3.5-flash


In [6]:
import time
from google import genai
from google.genai import types

client = genai.Client()  # reads GEMINI_API_KEY / GOOGLE_API_KEY from env

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the documents to find most the relevant paragraph.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=5):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=MODEL_NAME,
                contents=contents,
                config=config
            )
        except Exception as e:
            error = str(e)

            if "503" in error or "UNAVAILABLE" in error:
                if attempt < max_retries - 1:
                    wait = 30 * (attempt + 1)
                    print(f"Server busy. Retrying in {wait}s...")
                    time.sleep(wait)
                else:
                    raise
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_documents(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("What topics are students supposed to share documents on?")

Question: What topics are students supposed to share documents on?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'topics students share documents on'})
  Result preview: [Source: sports]

    Students can share documents about cricket, football and basketball.
    They ...
Step 2: model calls search_documents({'query': 'students share documents'})
  Result preview: [Source: sports]

    Students can share documents about cricket, football and basketball.
    They ...

Final answer:
Based on the documents, students can share documents on the following topics:

1. **Sports:**
   * **Sports covered:** Cricket, football, and basketball.
   * **Content details:** Match results, player performances, tournament records, team statistics, training techniques, and information about upcoming sporting events.

2. **Music:**
   * **Music covered:** Different music genres.
   * **Content details:** Artists, albums, musical instruments, music recomm

In [7]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.
questions = [
    "What topics are students supposed to share documents on?",
    "What information can students share about sports?",
    "What can students discuss about manga?",
    "What topics are covered in the music documents?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)



Question: What topics are students supposed to share documents on?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'topics students share documents on'})
  Result preview: [Source: sports]

    Students can share documents about cricket, football and basketball.
    They ...
Step 2: model calls search_documents({'query': 'Students can share documents about'})
  Result preview: [Source: sports]

    Students can share documents about cricket, football and basketball.
    They ...
Server busy. Retrying in 30s...

Final answer:
Based on the documents, students can share documents on the following topics:

1. **Sports:**
   * **Specific sports:** Cricket, football, and basketball.
   * **Content details:** Match results, player performances, tournament records, team statistics, training techniques, and information about upcoming sporting events.

2. **Music:**
   * **Specific areas:** Different music genres.
   * **Content details:** Artis

In [8]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)